<a href="https://colab.research.google.com/github/Ftaba-S4kura/attention-cnn-lstm-replica/blob/main/PROPUESTA_REPLICA_CAVERO_IAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_bb72ad5e03860ff234e8fcd7b3919d14"
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write("KGAT_bb72ad5e03860ff234e8fcd7b3919d14")
!chmod 600 /root/.kaggle/access_token

os.makedirs("datasets/bot_iot", exist_ok=True)
!kaggle datasets download -d vigneshvenkateswaran/bot-iot -p datasets/bot_iot --unzip
print("Listo.")

Dataset URL: https://www.kaggle.com/datasets/vigneshvenkateswaran/bot-iot
License(s): unknown
100% 1.17G/1.17G [00:15<00:00, 82.0MB/s]

Listo.


In [ ]:
import gc
import glob
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    LSTM, Dropout, Attention, GlobalAveragePooling1D, Dense
)
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, classification_report
)

print("GPU activa:", tf.config.list_physical_devices('GPU'))

def build_attention_cnn_lstm(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(filters=128, kernel_size=5, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    lstm_out = LSTM(units=64, return_sequences=True)(x)
    lstm_out = Dropout(rate=0.3)(lstm_out)

    attention_out = Attention(use_scale=True)([lstm_out, lstm_out])
    x = GlobalAveragePooling1D()(attention_out)

    x = Dense(64, activation='relu')(x)
    x = Dropout(rate=0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def run_pipeline(df, target_col, dataset_name):
    print(f"\n==================== {dataset_name} ====================")
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    X = pd.get_dummies(df.drop(columns=[target_col]), drop_first=True)
    le = LabelEncoder()
    y = le.fit_transform(df[target_col])
    num_classes = len(le.classes_)

    del df
    gc.collect()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    del X, y
    gc.collect()

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    model = build_attention_cnn_lstm((X_train.shape[1], 1), num_classes)
    model.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.1, verbose=1)

    start_time = time.time()
    y_pred = np.argmax(model.predict(X_test, batch_size=128), axis=1)
    total_time = time.time() - start_time

    acc = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    mcc = matthews_corrcoef(y_test, y_pred)

    print(f"\n[{dataset_name}] RESULTADOS:")
    print(f"Accuracy:  {acc:.2f}%")
    print(f"Precision: {prec:.2f}%")
    print(f"Recall:    {rec:.2f}%")
    print(f"F1-Score:  {f1:.2f}%")
    print(f"MCC:       {mcc:.4f}")
    print(f"Velocidad: {len(X_test)/total_time:.0f} reg/seg | Latencia: {(total_time/len(X_test))*1000:.4f} ms")
    print("\n", classification_report(y_test, y_pred, target_names=[str(c) for c in le.classes_]))

    del X_train, X_test, y_train, y_test
    gc.collect()
    return model

# ------------------------------------------------------------------------------
# EJECUCIÓN 1: NSL-KDD
# ------------------------------------------------------------------------------
col_names_nsl = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

df_nsl = pd.concat([
    pd.read_csv("datasets/nsl_kdd/KDDTrain+.txt", names=col_names_nsl),
    pd.read_csv("datasets/nsl_kdd/KDDTest+.txt", names=col_names_nsl)
], ignore_index=True).drop(columns=["difficulty"])

attack_map = {
    'normal': 'Normal',
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS',
    'teardrop': 'DoS', 'mailbomb': 'DoS', 'apache2': 'DoS', 'processtable': 'DoS', 'udpstorm': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L',
    'warezclient': 'R2L', 'warezmaster': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'snmpgetattack': 'R2L',
    'snmpguess': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L', 'worm': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
    'httptunnel': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R'
}

df_nsl['target'] = df_nsl['label'].map(attack_map)
df_nsl = df_nsl.drop(columns=['label'])
modelo_nsl = run_pipeline(df_nsl, 'target', 'NSL-KDD')



GPU activa: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

==================== NSL-KDD ====================
Epoch 1/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9609 - loss: 0.1240 - val_accuracy: 0.9767 - val_loss: 0.0679
Epoch 2/10
811/836 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9795 - loss: 0.0621

In [2]:
import gc, glob, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    LSTM, Dropout, Attention, GlobalAveragePooling1D, Dense
)
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, classification_report
)

# Cargar solo 4 archivos CSV para no saturar la RAM
bot_files = sorted(glob.glob("datasets/bot_iot/data_*.csv"))[:4]
df_bot = pd.concat((pd.read_csv(f, low_memory=False) for f in bot_files), ignore_index=True)

target_col = 'category' if 'category' in df_bot.columns else ('attack' if 'attack' in df_bot.columns else df_bot.columns[-1])
cols_to_drop = ['pkSeqID', 'stime', 'ltime', 'saddr', 'sport', 'daddr', 'dport', 'attack', 'subcategory', 'category', 'flgs', 'proto', 'state']
cols_to_drop = [c for c in cols_to_drop if c in df_bot.columns and c != target_col]

df_bot = df_bot.drop(columns=cols_to_drop).dropna(axis=1, how='all').fillna(0)
df_bot = df_bot.rename(columns={target_col: 'target'})

# Conservar solo columnas numéricas + target para evitar explosión de memoria en One-Hot
num_cols = df_bot.select_dtypes(include=[np.number]).columns.tolist()
df_bot = df_bot[num_cols + ['target']]

counts = df_bot['target'].value_counts()
df_bot = df_bot[df_bot['target'].isin(counts[counts >= 10].index)]

if len(df_bot) > 400000:
    df_bot, _ = train_test_split(df_bot, train_size=400000, stratify=df_bot['target'], random_state=42)

le = LabelEncoder()
y = le.fit_transform(df_bot['target'])
X = df_bot.drop(columns=['target']).values.astype(np.float32)
num_classes = len(le.classes_)

del df_bot
gc.collect()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
del X, y
gc.collect()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32).reshape((-1, X_train.shape[1], 1))
X_test = scaler.transform(X_test).astype(np.float32).reshape((-1, X_test.shape[1], 1))

# Modelo Attention-CNN-LSTM
inputs = Input(shape=(X_train.shape[1], 1))
x = Conv1D(64, 3, activation='relu', padding='same')(inputs)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)
x = Conv1D(128, 5, activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)
lstm_out = LSTM(64, return_sequences=True)(x)
lstm_out = Dropout(0.3)(lstm_out)
attention_out = Attention(use_scale=True)([lstm_out, lstm_out])
x = GlobalAveragePooling1D()(attention_out)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation='softmax')(x)

modelo_bot = Model(inputs, outputs)
modelo_bot.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo_bot.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.1, verbose=1)

start_time = time.time()
y_pred = np.argmax(modelo_bot.predict(X_test, batch_size=128), axis=1)
total_time = time.time() - start_time

print("\n[Bot-IoT] RESULTADOS:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0)*100:.2f}%")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0)*100:.2f}%")
print(f"MCC:       {matthews_corrcoef(y_test, y_pred):.4f}")
print(f"Velocidad: {len(X_test)/total_time:.0f} reg/seg | Latencia: {(total_time/len(X_test))*1000:.4f} ms")
print("\n", classification_report(y_test, y_pred, target_names=[str(c) for c in le.classes_]))

Epoch 1/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - accuracy: 0.9941 - loss: 0.0253 - val_accuracy: 0.9948 - val_loss: 0.0220
Epoch 2/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.9964 - loss: 0.0148 - val_accuracy: 0.9916 - val_loss: 0.0207
Epoch 3/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9969 - loss: 0.0132 - val_accuracy: 0.9968 - val_loss: 0.0129
Epoch 4/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9969 - loss: 0.0126 - val_accuracy: 0.9961 - val_loss: 0.0163
Epoch 5/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.9972 - loss: 0.0117 - val_accuracy: 0.9957 - val_loss: 0.0174
Epoch 6/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9972 - loss: 0.0113 - val_accuracy: 0.9969 - val_loss: 0.0135
Epoch 7/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.9973 - loss: 0.0107 - val_accuracy: 0.9910 - val_loss: 0.0282
Epoch 8/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9972 - loss: 0